In [2]:
import os
import sys
import glob
import numpy as np
import traceback
import cv2
from datetime import datetime
import psycopg2
from psycopg2.extras import Json
from insightface.app import FaceAnalysis

# 프로젝트 루트 경로 추가 (core 모듈 임포트용)
sys.path.append('../')
from core.config import ConfigLoader

# DB 연결 정보 (config.yaml에서 로드) - 노트북 기준 경로로 수정
db_cfg = ConfigLoader(config_path="../core/config.yaml").db
print("[DB CONFIG]", db_cfg)

conn = psycopg2.connect(
    host=db_cfg['host'], 
    port=db_cfg['port'], 
    user=db_cfg['user'], 
    password=db_cfg['password'], 
    dbname=db_cfg['dbname']
)
conn.autocommit = True
cur = conn.cursor()
print("DB 연결 완료")


[DB CONFIG] {'host': 'localhost', 'port': 5432, 'user': 'postgres', 'password': 'postgres', 'dbname': 'postgres'}
DB 연결 완료


In [3]:
# origin_vector 테이블 생성 (이미지+임베딩 일원화 테이블)
create_table_sql = '''
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS origin_vector (
    id serial PRIMARY KEY,
    image_path text NOT NULL UNIQUE,
    label text,
    vector_type varchar(32) NOT NULL,
    parameters json,
    embedding vector(512) NOT NULL,
    created_at timestamp,
    log text
);

-- 인덱스 생성 (검색 성능 향상)
CREATE INDEX IF NOT EXISTS idx_origin_vector_label ON origin_vector(label);
CREATE INDEX IF NOT EXISTS idx_origin_vector_vector_type ON origin_vector(vector_type);
'''

try:
    cur.execute(create_table_sql)
    print('origin_vector 테이블 및 인덱스 생성 완료')
except Exception as e:
    print(f'테이블 생성 에러: {e}')


origin_vector 테이블 및 인덱스 생성 완료


In [4]:
# ArcFace 모델 로드 (Buffalo_L 모델, CPU 기준)
print("ArcFace 모델 로딩 중...")
face_app = FaceAnalysis(name='buffalo_l', providers=['CPUExecutionProvider'])
face_app.prepare(ctx_id=0, det_size=(640, 640))
print('ArcFace 모델 로드 완료')

# ArcFace 파라미터 정의 (DB 저장용)
arcface_params = {
    'model': 'buffalo_l',
    'det_size': [640, 640],
    'provider': 'CPUExecutionProvider',
    'ctx_id': 0
}


ArcFace 모델 로딩 중...
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\pdc89/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\pdc89/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\pdc89/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\pdc89/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\pdc89/.insightface\models\buffalo_l\w600k_r50.onnx recognition ['None'

In [5]:
# downloaded_datasets 내 모든 이미지 파일 수집
root_dir = '../downloaded_datasets'  # 노트북 위치 기준 상대경로
image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff']
image_paths = []

for ext in image_extensions:
    image_paths.extend(glob.glob(os.path.join(root_dir, '*', ext)))
    image_paths.extend(glob.glob(os.path.join(root_dir, '*', ext.upper())))

print(f'총 이미지 수: {len(image_paths)}')
print(f'샘플 경로: {image_paths[:3] if image_paths else "이미지 없음"}')


총 이미지 수: 26466
샘플 경로: ['../downloaded_datasets\\Aaron_Eckhart\\Aaron_Eckhart_0001.jpg', '../downloaded_datasets\\Aaron_Guiel\\Aaron_Guiel_0001.jpg', '../downloaded_datasets\\Aaron_Patterson\\Aaron_Patterson_0001.jpg']


In [6]:
# 이미지별 임베딩 추출 및 DB 저장
processed_count = 0
error_count = 0
skipped_count = 0

print("임베딩 추출 및 저장 시작...")

for idx, img_path in enumerate(image_paths):
    # 상대경로로 변환 (DB 저장용)
    rel_path = os.path.relpath(img_path, start=root_dir)
    # 폴더명을 라벨로 사용
    label = os.path.basename(os.path.dirname(img_path))
    
    try:
        # 중복 방지: 이미 저장된 이미지면 건너뜀
        cur.execute("SELECT id FROM origin_vector WHERE image_path=%s", (rel_path,))
        if cur.fetchone():
            skipped_count += 1
            if idx % 100 == 0:
                print(f'진행률: {idx}/{len(image_paths)} (처리:{processed_count}, 에러:{error_count}, 스킵:{skipped_count})')
            continue
            
        # 이미지 로드 및 얼굴 검출
        img = cv2.imread(img_path)
        if img is None:
            raise ValueError("이미지 로드 실패")
            
        faces = face_app.get(img)
        
        if not faces or len(faces) == 0:
            # 얼굴 미검출 시 0벡터로 저장
            log_msg = 'No face detected'
            embedding = np.zeros(512, dtype=np.float32)
        else:
            # 첫 번째 얼굴의 임베딩 사용
            embedding = faces[0]['embedding']
            log_msg = f'Face detected, bbox: {faces[0]["bbox"]}'
        
        # DB 저장
        cur.execute(
            """INSERT INTO origin_vector 
               (image_path, label, vector_type, parameters, embedding, created_at, log) 
               VALUES (%s, %s, %s, %s, %s, %s, %s)""",
            (rel_path, label, 'origin', Json(arcface_params), embedding.tolist(), datetime.now(), log_msg)
        )
        
        processed_count += 1
        
        # 주기적 진행 상황 출력
        if idx % 100 == 0:
            print(f'진행률: {idx}/{len(image_paths)} (처리:{processed_count}, 에러:{error_count}, 스킵:{skipped_count})')
            
    except Exception as e:
        error_count += 1
        print(f'에러 [{idx}]: {rel_path} - {str(e)}')
        # 에러 로그도 DB에 저장 (임베딩은 0벡터)
        try:
            cur.execute(
                """INSERT INTO origin_vector 
                   (image_path, label, vector_type, parameters, embedding, created_at, log) 
                   VALUES (%s, %s, %s, %s, %s, %s, %s)""",
                (rel_path, label, 'origin', Json(arcface_params), np.zeros(512).tolist(), datetime.now(), f'ERROR: {str(e)}')
            )
        except:
            pass  # 중복 등으로 인한 DB 에러는 무시

print(f'\\n임베딩 추출 완료!')
print(f'- 총 이미지: {len(image_paths)}')
print(f'- 처리 완료: {processed_count}')
print(f'- 에러: {error_count}') 
print(f'- 스킵(중복): {skipped_count}')


임베딩 추출 및 저장 시작...


d:\ronbun\.conda\lib\site-packages\insightface\utils\transform.py:68: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  P = np.linalg.lstsq(X_homo, Y)[0].T # Affine matrix. 3 x 4


진행률: 0/26466 (처리:1, 에러:0, 스킵:0)
진행률: 100/26466 (처리:101, 에러:0, 스킵:0)
진행률: 200/26466 (처리:201, 에러:0, 스킵:0)
진행률: 300/26466 (처리:301, 에러:0, 스킵:0)
진행률: 400/26466 (처리:401, 에러:0, 스킵:0)
진행률: 500/26466 (처리:501, 에러:0, 스킵:0)
진행률: 600/26466 (처리:601, 에러:0, 스킵:0)
진행률: 700/26466 (처리:701, 에러:0, 스킵:0)
진행률: 800/26466 (처리:801, 에러:0, 스킵:0)
진행률: 900/26466 (처리:901, 에러:0, 스킵:0)
진행률: 1000/26466 (처리:1001, 에러:0, 스킵:0)
진행률: 1100/26466 (처리:1101, 에러:0, 스킵:0)
진행률: 1200/26466 (처리:1201, 에러:0, 스킵:0)
진행률: 1300/26466 (처리:1301, 에러:0, 스킵:0)
진행률: 1400/26466 (처리:1401, 에러:0, 스킵:0)
진행률: 1500/26466 (처리:1501, 에러:0, 스킵:0)
진행률: 1600/26466 (처리:1601, 에러:0, 스킵:0)
진행률: 1700/26466 (처리:1701, 에러:0, 스킵:0)
진행률: 1800/26466 (처리:1801, 에러:0, 스킵:0)
진행률: 1900/26466 (처리:1901, 에러:0, 스킵:0)
진행률: 2000/26466 (처리:2001, 에러:0, 스킵:0)
진행률: 2100/26466 (처리:2101, 에러:0, 스킵:0)
진행률: 2200/26466 (처리:2201, 에러:0, 스킵:0)
진행률: 2300/26466 (처리:2301, 에러:0, 스킵:0)
진행률: 2400/26466 (처리:2401, 에러:0, 스킵:0)
진행률: 2500/26466 (처리:2501, 에러:0, 스킵:0)
진행률: 2600/26466 (처리:2601, 에러:0, 스킵:0

In [7]:
# 결과 확인 및 샘플 데이터 조회
print("=== 데이터베이스 저장 결과 확인 ===")

# 전체 레코드 수
cur.execute("SELECT COUNT(*) FROM origin_vector")
total_count = cur.fetchone()[0]
print(f"총 저장된 레코드 수: {total_count}")

# 라벨별 통계
cur.execute("""
    SELECT label, COUNT(*) as count 
    FROM origin_vector 
    GROUP BY label 
    ORDER BY count DESC 
    LIMIT 10
""")
print("\\n상위 10개 라벨별 이미지 수:")
for label, count in cur.fetchall():
    print(f"  {label}: {count}개")

# 얼굴 검출 성공/실패 통계
cur.execute("""
    SELECT 
        CASE 
            WHEN log LIKE 'Face detected%' THEN 'Face Detected'
            WHEN log = 'No face detected' THEN 'No Face'
            ELSE 'Error'
        END as status,
        COUNT(*) as count
    FROM origin_vector 
    GROUP BY 
        CASE 
            WHEN log LIKE 'Face detected%' THEN 'Face Detected'
            WHEN log = 'No face detected' THEN 'No Face'
            ELSE 'Error'
        END
""")
print("\\n얼굴 검출 결과 통계:")
for status, count in cur.fetchall():
    print(f"  {status}: {count}개")

# 샘플 데이터 조회
cur.execute("""
    SELECT id, image_path, label, 
           CASE 
               WHEN log LIKE 'Face detected%' THEN 'OK'
               ELSE log
           END as status
    FROM origin_vector 
    ORDER BY id 
    LIMIT 5
""")
print("\\n샘플 데이터 (처음 5개):")
for row in cur.fetchall():
    print(f"  ID:{row[0]}, Path:{row[1]}, Label:{row[2]}, Status:{row[3]}")

print("\\n=== 작업 완료 ===")


=== 데이터베이스 저장 결과 확인 ===
총 저장된 레코드 수: 13233
\n상위 10개 라벨별 이미지 수:
  George_W_Bush: 530개
  Colin_Powell: 236개
  Tony_Blair: 144개
  Donald_Rumsfeld: 121개
  Gerhard_Schroeder: 109개
  Ariel_Sharon: 77개
  Hugo_Chavez: 71개
  Junichiro_Koizumi: 60개
  Jean_Chretien: 55개
  John_Ashcroft: 53개
\n얼굴 검출 결과 통계:
  No Face: 38개
  Face Detected: 13195개
\n샘플 데이터 (처음 5개):
  ID:1, Path:Aaron_Eckhart\Aaron_Eckhart_0001.jpg, Label:Aaron_Eckhart, Status:OK
  ID:2, Path:Aaron_Guiel\Aaron_Guiel_0001.jpg, Label:Aaron_Guiel, Status:OK
  ID:3, Path:Aaron_Patterson\Aaron_Patterson_0001.jpg, Label:Aaron_Patterson, Status:OK
  ID:4, Path:Aaron_Peirsol\Aaron_Peirsol_0001.jpg, Label:Aaron_Peirsol, Status:OK
  ID:5, Path:Aaron_Peirsol\Aaron_Peirsol_0002.jpg, Label:Aaron_Peirsol, Status:OK
\n=== 작업 완료 ===


In [8]:
# DB 연결 종료
cur.close()
conn.close()
print("DB 연결 종료")


DB 연결 종료
